# Análisis de distribución de la popularidad (número de reseñas en el primer mes)

El objetivo de este notebook es analizar la distribución del número de reseñas de los juegos en Steam

---
## Importación de librerías

In [ ]:
# Módulo
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Para que se puedan utilizar funciones desde el notebook
from src.utils.files import read_file
from src.utils.config import popularity, load_env_file
load_env_file()

## Carga de datos
Configuración del Minio, poner en True para usar la información de Minio

In [ ]:
use_minio = True
minio = {"minio_write": False, "minio_read": use_minio}

In [ ]:
df = read_file(popularity, minio)
df.head(2)

---

## Análisis

Primero visualizaremos los datos sin modificar

### Histograma del número de reseñas

In [ ]:
def muestra_data(df, titulo):
    fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("Escala Lineal", "Escala Logarítmica"),
    horizontal_spacing=0.12
    )

    trace = go.Histogram(
        x=df["recomendaciones_totales"],
        nbinsx=200,
        marker_color="teal",
        showlegend=False
    )

    fig.add_trace(trace, row=1, col=1)
    fig.add_trace(trace, row=1, col=2)

    fig.update_yaxes(type="log", row=1, col=2)
    fig.update_layout(
        width=1100, 
        height=550,
        font_color="white",
        title_text=f"<b>Distribución de popularidad {titulo}</b>",
        title_x=0.5,
        title_xanchor='center',
        plot_bgcolor="powderblue",
        paper_bgcolor="#131416"
    )

    fig.update_xaxes(title_text="Número de reseñas", gridcolor="white", tickfont=dict(color="white"))
    fig.update_yaxes(title_text="Número de juegos", gridcolor="white", tickfont=dict(color="white"))

    fig.update_yaxes(title_text="Número de juegos (log)", row=1, col=2)
    fig.show()

In [ ]:
muestra_data(df, "con todos los datos")

Se observa que la distribución del número de reseñas es una distribución de cola larga en la que el 80% de los datos no llegan ni a 50 reseñas, específicamente:

In [ ]:
porcentaje = round((df[df["recomendaciones_totales"] > 200].shape[0] / df.shape[0] * 100), 4)
print(f"El {porcentaje}% de los juegos tiene más de 200 reseñas")

porcentaje = round((df[df["recomendaciones_totales"] < 50].shape[0] / df.shape[0] * 100), 4)
print(f"El {porcentaje}% de los juegos no llegan a 50 reseñas")

maximos = round((df[df["recomendaciones_totales"] > 2000].shape[0] / df.shape[0] * 100), 4)
print(f"El {maximos}% de los juegos tiene más de 2000 reseñas")

Como hemos visto, tan solo el 6,75% de los datos tienen más de 200 reseñas, por lo que ahora visualizaremos el otro 93,25% de los datos

In [ ]:
df_clean1 = df[df["recomendaciones_totales"] <= 200]
muestra_data(df_clean1, "limpiando outliers")

Podemos intentar ver si hay algún cambio al quedarnos solo con los videos que tienen un popularity score mayor a 0.5, esto quiere decir que la métrica obtenida procesando con PCA la información de YouTube (visualizaciones, likes, comentarios...) indica que el juego tiene algo de impacto mediático. 

In [ ]:
df_clean2 = df[df["yt_score"] >= 0.5]
muestra_data(df_clean2, "con buen popularity_score")

Ahora vamos a intentar analizar si la distribución es diferente en diferentes grupos, en este caso por precios:

### Histograma del número de reseñas por rango de precio para menos de 200 reseñas

In [ ]:
df_cleaned = df[(df["recomendaciones_totales"] <= 50000) & (df["recomendaciones_totales"] <= 200)]

dfs = [
    df_cleaned[df_cleaned["price_overview"] == 0],
    df_cleaned[(df_cleaned["price_overview"] >= 0.1) & (df_cleaned["price_overview"] < 5)],
    df_cleaned[(df_cleaned["price_overview"] >= 5) & (df_cleaned["price_overview"] < 10)],
    df_cleaned[(df_cleaned["price_overview"] >= 10) & (df_cleaned["price_overview"] < 15)],
    df_cleaned[(df_cleaned["price_overview"] >= 15) & (df_cleaned["price_overview"] < 20)],
    df_cleaned[(df_cleaned["price_overview"] >= 20) & (df_cleaned["price_overview"] < 40)],
    df_cleaned[df_cleaned["price_overview"] >= 40]
]

titulos = ["Gratis", "0.1-5€", "5-10€", "10-15€", "15-20€", "20-40€", "40€+"]

fig = make_subplots(
    rows=1, cols=len(dfs), 
    subplot_titles=titulos,
    shared_yaxes=True,
    horizontal_spacing=0.02
)

for i, subset in enumerate(dfs):
    fig.add_trace(
        go.Histogram(
            x=subset["recomendaciones_totales"], 
            nbinsx=100, 
            marker_color="teal",
            showlegend=False
        ),
        row=1, col=i+1
    )

fig.update_yaxes(type="log")

fig.update_layout(
    width = 1600,
    height = 550,
    font_color = "white",
    title_text="<b>Distribución de popularidad por tramos de precio</b>",
    title_x=0.5,
    title_xanchor='center',
    plot_bgcolor = "powderblue",
    paper_bgcolor = "#131416"
    )

fig.update_xaxes(
    title_text = "Reseñas", 
    gridcolor = "white",
    tickfont = dict(color="white") 
)

fig.update_yaxes(
    gridcolor = "white",
    tickfont = dict(color="white")
)

fig.update_yaxes(title_text = "Número de juegos (log)", row=1, col=1)

fig.show()

---

## Conclusión

Se aprecia que la distribución era la que esperábamos, una de cola larga con gran concentración de juegos alrededor de números de reseñas bajos y solo unos pocos con muchas reseñas. Además también se comprueba que la distribución se mantiene tanto para juegos gratuitos como no gratuitos, así como para los juegos con diferente repercusión en YouTube.

In [ ]:
df_cleaned = df.copy()
df_cleaned["viewCountTotal"] = df_cleaned.filter(like="video_statistics.viewCount").sum(axis=1)
df_cleaned["likeCountTotal"] = df_cleaned.filter(like="video_statistics.likeCount").sum(axis=1)
df_cleaned["commentCountTotal"] = df_cleaned.filter(like="video_statistics.commentCount").sum(axis=1)

numeric_columns = ["description_len", "price_overview", "num_languages", "num_juegos_previos_developers", "ema_reviews_developers", "max_historico_reviews_developers", \
            "num_juegos_previos_publishers", "ema_reviews_publishers", "max_historico_reviews_publishers", "brillo", "viewCountTotal", \
            "likeCountTotal", "commentCountTotal"]

for col in numeric_columns:
    fig = px.histogram(df_cleaned, x=col, title=f"Distribucion de {col}")
    fig.show()

La mayoría de nuestras variables numéricas, al igual que nuestra variable respuesta, siguen una distribución de cola larga. <br>
En la variable `description_len` se ve que a partir de los 300 caracteres cambia la distribución de los datos, esto es porque 300 es el límite establecido por Steam. Los caracteres extra son probablemente de tags html y formateadores como `\t`.

---
### Mapa de correlación

In [ ]:
corr = df_cleaned[numeric_columns].corr(method='spearman')
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', annot=True)
plt.show()

In [ ]:
df_cleaned.columns

En general no hay mucha correlación entre la variables numéricas, excepto en las siguientes: <br>
- Estadístico ema y máximo histórico entre developers y publishers: tiene sentido porque ambos representan casi lo mismo excepto en ventanas de tiempo diferentes.
- Estadísticos de developers y publishers: esta relación es un poco menor, pero también es lógico, muchas veces developer y publisher son los mismos.
- Estadísticas de youtube: es normal que tenga más likes y comentarios un video con más visualizaciones.

In [ ]:

upper_limit = df_cleaned["recomendaciones_totales"].quantile(0.99)

filtro = df_cleaned["recomendaciones_totales"] < upper_limit
upper_limit2 = df_cleaned["price_overview"].quantile(0.99)
filtro2 = df_cleaned["price_overview"] < upper_limit2
filtro3 = df_cleaned["price_overview"] > 0
df_trimmed = df_cleaned[filtro & filtro2 & filtro3]
fig = px.scatter(df_trimmed, x='price_overview', y='recomendaciones_totales',
                 title='Price vs Recommendations',
                 hover_data=['name'])
fig.show()


No se ve una relación clara entre precio y reseñas, lo que sí que se puede observar es las agrupaciones de precios, las columnas que se pueden distinguir en este gráfico son los clusters de precios que existe, por ejemplo 9.99€, 19.99€ etc.

In [ ]:
fig = px.histogram(df, x='release_year', title='Games per Release Year')
fig.show()

La tendencia muestra el crecimiento de la industria de los videojuegos, cada año aumenta el número de juegos publicados en lo que parece una tendencia lineal. 2026 es un año incompleto

In [ ]:
genre_cols = ['Action','Adventure','Casual','Indie','RPG','Simulation','Strategy']
genre_counts = df[genre_cols].sum().sort_values(ascending=False)
fig = px.bar(x=genre_counts.index, y=genre_counts.values,
             title='Genre Distribution')
fig.show()

In [ ]:
fig = px.scatter(df_trimmed,
                 x='num_juegos_previos_developers',
                 y='recomendaciones_totales',
                 title='Developer Experience vs Recommendations')
fig.show()

In [ ]:
fig = px.scatter(df_trimmed,
                 x='num_juegos_previos_publishers',
                 y='recomendaciones_totales',
                 title='Publisher Experience vs Recommendations')
fig.show()

In [ ]:
fig = px.scatter(df_trimmed, x='brillo', y='recomendaciones_totales',
                 title='Brightness vs Recommendations')
fig.show()

In [ ]:
fig = px.scatter(df_trimmed, x='num_languages', y='recomendaciones_totales',
                 title='Languages vs Recommendations')
fig.show()

En ninguna de estas gráficas se aprecia una relación clara con la variable respuesta de recomendaciones totales

In [ ]:
df_trimmed["release_year"].value_counts()

In [ ]:
year_group = df_trimmed.groupby('release_year')['recomendaciones_totales'].median().reset_index()
fig = px.line(year_group, x='release_year', y='recomendaciones_totales',
              title='Median Recommendations Over Time')
fig.show()

En esta gráfica se observa algo curioso y es cómo ha ido variando el número de reseñas en los juegos a lo largo del tiempo. Se observa que en los primero años, no había tantos juegos en Steam, el mercado estaba dominado por aquellos juegos que Steam decidía y los que vendían bien, es por esto que la mediana de reseñas es tan alta. <br>
Después, con el lanzamiento de Steam Greenlight y posteriormente Steam Direct,se ha aumentado la accesibilidad para publicar en Steam a muchos más títulos, provocando saturación en el mercado y la existencia de muchos juegos con pocas reseñas. <br>
La razón por la que probablemente la tendencia haya cambiado estos últimos años es por:
- El periodo de pandemia de COVID-19, que ha dado mayor visibilidad al mundo de los videojuegos y en general a la comunidad online
- Los desarrolladores están mejorando en el aspecto de conseguir feedback de los usuarios. Por ejemplo, mediante los típicos pop-ups que te preguntan si te está gustando el juego y que dejes una reseña, o recompensas in-game por dejar una reseña

In [ ]:
top_games = df.sort_values(by='recomendaciones_totales', ascending=False).head(20)
fig = px.bar(top_games, x='name', y='recomendaciones_totales',
             title='Top 20 Games by Recommendations')
fig.show()


In [ ]:
import pandas as pd
from scipy.stats import spearmanr

correlaciones = df_cleaned[numeric_columns].corrwith(
    df_cleaned['recomendaciones_totales'], method='spearman').sort_values() # Usamos spearman porque no asume normalidad

fig1 = go.Figure(go.Bar(
    x=correlaciones.values,
    y=correlaciones.index,
    orientation='h',
    marker_color=['#FFA43B' if x < 0 else "#3BA4FF" for x in correlaciones.values]
))
fig1.add_vline(x=0, line_width=1, line_color='black')
fig1.update_layout(title="Correlación con recomendaciones_totales", template='plotly_white', height=700)
fig1.show()

resultados = {}
for col in numeric_columns:
    r, p = spearmanr(df_cleaned[col], df_cleaned['recomendaciones_totales'])
    resultados[col] = {'r (coeficiente de Spearman)': r, 'p-valor': p}

pd.DataFrame(resultados).T.sort_values('r (coeficiente de Spearman)', ascending=False)

In [ ]:
fig = go.Figure()

for col in numeric_columns + ['recomendaciones_totales']:
    fig.add_trace(go.Box(
        x=df_cleaned[col],
        name=col,
        orientation='h'
    ))

fig.update_layout(
    title="Outliers en variables continuas (Escala Logarítmica)",
    xaxis=dict(type='log'),
    plot_bgcolor='white',
    showlegend=False,
    height=100 + len(numeric_columns) * 55
)

fig.show()

In [ ]:
extreme_skews = []
extreme_kurtosis = []

df_numeric = df_cleaned[numeric_columns + ['recomendaciones_totales']]

print(f"{'COLUMNA':<32} | {'SKEW':>10} | {'CURTOSIS':>10}")
print("-" * 60)

for col in df_numeric.columns:
    skew = df_numeric[col].skew()
    kurtosis = df_numeric[col].kurtosis()

    print(f"{col:<32} | {skew:>10.3f} | {kurtosis:>10.3f}")

    if skew < -1 or skew > 1:
        extreme_skews.append(col)
    if kurtosis > 2: 
        extreme_kurtosis.append(col)


print(f"\nSkew extremo:      {', '.join(extreme_skews) if extreme_skews else 'Ninguno'}")
print(f"Curtosis extrema:  {', '.join(extreme_kurtosis) if extreme_kurtosis else 'Ninguno'}")

In [ ]:
df_cleaned.columns

In [ ]:
df_cleaned.recomendaciones_totales.value_counts().head(20)

In [ ]:
df_nulos = pd.DataFrame({
    'Total Nulos': df_cleaned.isnull().sum(),
    '% Nulos': round(df_cleaned.isnull().mean() * 100, 2)
}).sort_values('Total Nulos', ascending=False)

display(df_nulos[df_nulos['Total Nulos'] > 0])

plt.figure(figsize=(14, 6), facecolor='white')
sns.heatmap(df_cleaned.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title("Mapa de Nulos", color='black', fontsize=14)
plt.xticks(color='black', rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
display(df[df.isnull().any(axis=1)][["name", "num_juegos_previos_publishers", "num_juegos_previos_developers"]])

In [ ]:
df_cleaned = df_cleaned.copy()
df_cleaned = df_cleaned.dropna(subset=["num_juegos_previos_developers", "num_juegos_previos_publishers"], how="all")

cols_devs = [
    "num_juegos_previos_developers", 
    "es_primer_juego_developers", 
    "ema_reviews_developers", 
    "max_historico_reviews_developers"
]

cols_pubs = [
    "num_juegos_previos_publishers", 
    "es_primer_juego_publishers", 
    "ema_reviews_publishers", 
    "max_historico_reviews_publishers"
]

filtro_devs = df_cleaned["num_juegos_previos_developers"].isna()
filtro_pubs = df_cleaned["num_juegos_previos_publishers"].isna()

display(df_cleaned[filtro_devs & filtro_pubs])

df_cleaned.loc[filtro_devs, cols_devs] = df_cleaned.loc[filtro_devs, cols_pubs].values
df_cleaned.loc[filtro_pubs, cols_pubs] = df_cleaned.loc[filtro_pubs, cols_devs].values

In [ ]:
df_nulos = pd.DataFrame({
    'Total Nulos': df_cleaned.isnull().sum(),
    '% Nulos': round(df_cleaned.isnull().mean() * 100, 2)
}).sort_values('Total Nulos', ascending=False)

display(df_nulos[df_nulos['Total Nulos'] > 0])

plt.figure(figsize=(14, 6), facecolor='white')
sns.heatmap(df_cleaned.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title("Mapa de Nulos", color='black', fontsize=14)
plt.xticks(color='black', rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
df_v_clip = pd.DataFrame(df_cleaned['v_clip'].tolist())
print(df_v_clip)
df_nulos = pd.DataFrame({
    'Total Nulos': df_v_clip.isnull().sum(),
    '% Nulos': round(df_v_clip.isnull().mean() * 100, 2)
}).sort_values('Total Nulos', ascending=False)

display(df_nulos[df_nulos['Total Nulos'] > 0])

plt.figure(figsize=(14, 6), facecolor='white')
sns.heatmap(df_v_clip.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title("Mapa de Nulos", color='black', fontsize=14)
plt.xticks(color='black', rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
df_cleaned['v_clip']

In [ ]:
print(df.columns)

In [ ]:
import numpy as np

y = df['recomendaciones_totales']
p99 = np.percentile(y, 99)
print(f"El 99% de los juegos tienen {p99:.2f} recomendaciones o menos.")
print(f"El juego más viral tiene {y.max()} recomendaciones.")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(y, bins=50, ax=axes[0])
axes[0].set_title("Target Original (Bruto)")

sns.histplot(np.log1p(y), bins=50, ax=axes[1], color='green')
axes[1].set_title("Target con np.log1p")

sns.histplot(y.clip(upper=p99), bins=50, ax=axes[2], color='orange')
axes[2].set_title(f"Target Capeado (Max: {p99:.0f})")

plt.show()

In [ ]:
df['views'] = df.filter(like="viewCount").sum(axis=1).replace(0, 1)
df['likes'] = df.filter(like="likeCount").sum(axis=1)
df['comments'] = df.filter(like="commentCount").sum(axis=1)

df['tasa_likes'] = df['likes'] / df['views']
df['tasa_comments'] = df['comments'] / df['views']

cols_to_check = ['views', 'likes', 'comments', 'tasa_likes', 'tasa_comments', 'recomendaciones_totales']
corr = df[cols_to_check].corr(method='spearman')

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlación de Spearman (Nuevas Variables vs Target)")
plt.show()

In [ ]:
import umap

print("Extrayendo vectores...")
zero_vector = np.zeros(512)
v_clip_clean = df['v_clip'].apply(lambda x: x if isinstance(x, (list, np.ndarray)) else zero_vector)
clip_matrix = np.vstack(v_clip_clean.values)

print("Aplicando UMAP (esto tomará unos segundos/minutos)...")
reducer = umap.UMAP(n_components=10, random_state=42)
clip_reduced = reducer.fit_transform(clip_matrix)

columnas_umap = [f'UMAP_{i}' for i in range(10)]
df_umap = pd.DataFrame(clip_reduced, columns=columnas_umap)

df_umap['target_log'] = np.log1p(df['recomendaciones_totales'].values)

plt.figure(figsize=(12, 8))
corr_umap = df_umap.corr(method='spearman')

sns.heatmap(corr_umap, annot=True, cmap='coolwarm', fmt=".2f", center=0)
plt.title("Correlación de Spearman: Componentes UMAP vs Target (Log)")
plt.show()

In [ ]:
df['recomendaciones_totales'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
resultados = []
df_prueba = df.drop(columns=['v_clip', 'v_resnet', 'v_convnext'])
binarias = [col for col in df_prueba.columns if df_prueba[col].nunique() == 2]
for col in binarias:
    conteos = df_prueba[col].value_counts()
    if not conteos.empty:
        resultados.append({
            'Columna': col,
            'Valor_menos_frecuente': conteos.idxmin(),
            'Apariciones': conteos.min()
        })

df_resultado = pd.DataFrame(resultados).sort_values('Apariciones')
print(df_resultado[df_resultado['Apariciones'] < 100])
print()

df_filtro = df_resultado[df_resultado['Apariciones'] < 100]

for col, val in zip(df_filtro['Columna'], df_filtro['Valor_menos_frecuente']):
    print("-" * 40)
    print(f"JUEGOS CON {col} = {val}")
    print("-" * 40)
    print(df[df[col] == val][["name", "price_range", "price_overview"]])
    print()

In [ ]:
print(df[df['Free To Play'] == True][['name', 'recomendaciones_totales', 'price_overview']])